# Volleyball court keypoint model v3 (YOLO pose)

Trains the 14-point court keypoint detector (see `src/vball/court_keypoints.py`) on the zip built by
`scripts/build_court_dataset.py` (v3): k6y7r for train/valid/test, plus VNL broadcast clips one-three
(front-zone labels completed to all floor points) in train, and VNL clips four-five held out as `test_vnl`.
Decisions and measurements: `openspec/changes/add-court-registration/design.md`, `docs/results/court-keypoints.md`.

1. Runtime -> Change runtime type -> **T4 GPU**
2. Run all cells. The best weights are copied back to `MyDrive/volleyball/models/court_kpt_yolo26s_v3.pt`.
3. If Colab disconnects, just **Run all** again: checkpoints are saved to
   `MyDrive/volleyball/runs/court_kpt_v3/` every epoch and training resumes from `last.pt`.
4. Out of GPU memory at the start? Change `batch=8` to `batch=4` in the training cell (only before the first epoch).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# Dataset zip uploaded from the laptop to MyDrive/volleyball/datasets/
DATASET_ZIP = '/content/drive/MyDrive/volleyball/datasets/court_keypoints_v3.zip'
!rm -rf /content/data && mkdir -p /content/data
!unzip -q "$DATASET_ZIP" -d /content/data
!find /content/data -maxdepth 2 -name data.yaml

In [ ]:
import glob, yaml, os
cfg = sorted(glob.glob('/content/data/**/data.yaml', recursive=True))[0]
root = os.path.dirname(cfg)
d = yaml.safe_load(open(cfg))
d['path'] = root
# Roboflow exports relative paths; point them at the unzipped folder
for split in ('train', 'val', 'test'):
    if split in d:
        name = 'valid' if split == 'val' else split
        d[split] = os.path.join(root, name, 'images')
# extra split, read by model.val(split='test_vnl')
d['test_vnl'] = os.path.join(root, 'test_vnl', 'images')
yaml.safe_dump(d, open(cfg, 'w'))
print(cfg); print(d)

In [ ]:
from ultralytics import YOLO
# Checkpoints go to Drive so a Colab disconnect does not lose training; rerunning resumes.
RUNS = '/content/drive/MyDrive/volleyball/runs'
RUN_NAME = 'court_kpt_v3'
last = f'{RUNS}/{RUN_NAME}/weights/last.pt'

import torch
# last.pt has epoch == -1 once training finished (or stopped early): then skip, don't retrain.
# (ultralytics would otherwise silently start a brand-new run in runs/pose/train-N.)
finished = os.path.exists(last) and torch.load(last, map_location='cpu', weights_only=False).get('epoch') == -1

if finished:
    print('training already finished:', last)
elif os.path.exists(last):
    print('resuming from', last)
    model = YOLO(last)
    model.train(resume=True)
else:
    # v3: small model at 1280 px (v2 nano at 960 px was still improving at epoch 300; far lines are thin)
    model = YOLO('yolo26s-pose.pt')
    model.train(data=cfg, epochs=400, imgsz=1280, batch=8, patience=80,
                fliplr=0.5,           # data.yaml flip_idx follows the image-based numbering (tests/test_court_keypoints.py)
                mosaic=0.5, degrees=3, perspective=0.0005,
                project=RUNS, name=RUN_NAME, exist_ok=True,
                save_period=10)       # also keep epoch10.pt, epoch20.pt, ... besides last.pt / best.pt


In [ ]:
model = YOLO(f'{RUNS}/{RUN_NAME}/weights/best.pt')
metrics = model.val(data=cfg, split='test')
print('k6y7r test  pose mAP50-95:', metrics.pose.map, ' mAP50:', metrics.pose.map50)
# Held-out VNL clips four-five (same broadcast as the training clips, never trained on)
vnl = model.val(data=cfg, split='test_vnl')
print('VNL extern  pose mAP50-95:', vnl.pose.map, ' mAP50:', vnl.pose.map50)
import shutil
run_dir = f'{RUNS}/{RUN_NAME}'
dst = '/content/drive/MyDrive/volleyball/models'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'{run_dir}/weights/best.pt', f'{dst}/court_kpt_yolo26s_v3.pt')
shutil.copy(f'{run_dir}/results.png', f'{dst}/court_kpt_v3_results.png')
print('saved to', dst)